# Lab 5.3 · Numerically Stable QR

Measure reconstruction and orthogonality separately on an input with small singular values.

Wooseok Ha, MAS110 Lecture 5, printed slides 1, 23, and 30, and the QR construction in Lecture 4. The comparison of algorithms is a coding extension.

Almost parallel columns force Gram–Schmidt to subtract nearly equal vectors. The small remainders can lose significant digits. Normalizing a remainder can magnify the error in its direction.

Run the cells in order with NumPy. The code and notation follow the web lab.

## Notation

- $A=QR$ · Here $Q$ has three orthonormal columns in exact arithmetic and $R$ is upper triangular. Both Gram–Schmidt variants build these factors column by column.

- $\kappa_2(A)=\sigma_1/\sigma_3$ · For this full-column-rank example, the ratio compares the largest and smallest stretches. It extends the invertible square case on slide 30 to a rectangular matrix with independent columns.

- $Q^{\top}Q-I$ · Diagonal errors measure unit length and off-diagonal entries measure unwanted alignment.

## 1. Measure the small stretches

The last two singular values are small. In exact arithmetic, $A^{\top}A$ has $\varepsilon^2$ on top of the all-ones matrix’s diagonal.

**Predict first.** Is the exact rank one or three?

In [ ]:
import numpy as np
eps = 1e-8
A = np.vstack((np.ones((1, 3)), eps * np.eye(3)))
s = np.linalg.svd(A, compute_uv=False)
print("Singular values =", s)

The largest stretch is about 173 million times the smallest.

## 2. Change the coefficient source

The two variants differ in which vector supplies the coefficient. The function below is used with the fixed independent input above.

**Predict first.** Which version recomputes a coefficient after the previous subtraction?

In [ ]:
def gram_schmidt(A, modified):
    n, d = A.shape
    Q, R = np.zeros((n, d)), np.zeros((d, d))
    for j in range(d):
        w = A[:, j].copy()
        for i in range(j):
            reference = w if modified else A[:, j]
            R[i, j] = Q[:, i] @ reference
            w -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(w)
        if R[j, j] == 0:
            raise ValueError("Zero Gram-Schmidt remainder")
        Q[:, j] = w / R[j, j]
    return Q, R

The function is defined. The next step runs it twice and compares the results with library QR.

## 3. Measure the two errors separately

All three methods reconstruct the matrix closely. Classical Gram–Schmidt loses much more perpendicularity on this input.

**Predict first.** Which of the two error measures separates the methods?

In [ ]:
factors = [gram_schmidt(A, False), gram_schmidt(A, True), np.linalg.qr(A)]
for Q, R in factors:
    reconstruction = np.linalg.norm(A - Q @ R) / np.linalg.norm(A)
    orthogonality = np.linalg.norm(Q.T @ Q - np.eye(3))
    print(f"{reconstruction:.2e} {orthogonality:.2e}")

The order is classical, modified, and library QR. The entries below are approximate scales and can vary with floating-point libraries.

## Why a small reconstruction error can hide a bad basis

### Use the right vector for each coefficient

Classical Gram–Schmidt takes every coefficient for column $j$ against the original $a_j$. Modified Gram–Schmidt takes each coefficient against the current remainder $w$. Earlier directions have already been subtracted from that remainder. In exact arithmetic the coefficients agree because the accepted columns are mutually orthogonal.

### Check two different promises of QR

The relative reconstruction error tests the product. The orthogonality error tests whether the computed columns can be used as orthonormal directions, for example in the projection $QQ^{\top}$. An accurate product can coexist with large off-diagonal entries in $Q^{\top}Q$. The library QR here uses Householder reflectors, which act on the trailing matrix by orthogonal transformations.

$$\frac{\lVert A-QR\rVert_F}{\lVert A\rVert_F},\quad \lVert Q^{\top}Q-I\rVert_F$$

## Verify the example

Run these checks on the example before changing the input.

In [ ]:
assert np.allclose(s, [np.sqrt(3), eps, eps], rtol=1e-6, atol=0)
orthogonality_errors = []
for Q, R in factors:
    assert np.linalg.norm(A - Q @ R) / np.linalg.norm(A) < 1e-12
    orthogonality_errors.append(np.linalg.norm(Q.T @ Q - np.eye(3)))
assert orthogonality_errors[0] > 1e-3
assert orthogonality_errors[1] < orthogonality_errors[0]
assert orthogonality_errors[2] < 1e-12

## Check your understanding

If $QR$ reconstructs $A$ accurately, is $QQ^{\top}$ necessarily a good projector?

Check $Q^{\top}Q$ separately. Nearly parallel computed columns can produce an accurate product but make $QQ^{\top}$ far from an orthogonal projector.

What happens to exact rank when the small entries are set to zero?

All three columns become equal, so the rank drops from three to one. A zero remainder cannot be normalized. This walkthrough uses a fixed nonzero perturbation.

## References

- [NumPy QR](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html)
- [Original Chapter 5.3 notebook](https://github.com/kyunghyuncho/Foundations_of_LADS/blob/main/Ch5-3%20Numerically%20Stable%20QR-Decomposition.ipynb)